In [2]:
import pandas as pd
import numpy as np
import zipfile
import os
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv('project_dataset_housing.csv')
print(df.head())

df.shape


      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  


(545, 13)

In [3]:
#OLS Model 

# preprocessing of data

BINARY_COLS = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]

FURNISHING_ORDER = {"unfurnished":  0, "semi-furnished": 1, "furnished":  2}

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in BINARY_COLS:
        df[col] = (df[col].str.strip().str.lower() == "yes").astype(int)

    df["furnishingstatus"] = (df["furnishingstatus"].str.strip().str.lower().map(FURNISHING_ORDER))
    return df


#  training / testing split of data

df_clean = preprocess(df)
df_clean.sample(10)

X = df_clean.drop("price", axis=1)
y = df_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [24]:
#linear regression model
ols = LinearRegression()
ols.fit(X_train, y_train)
y_pred = ols.predict(X_test)
mse_ols = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse_ols)
r2 = r2_score(y_test, y_pred)
coef_df = pd.DataFrame({"Feature": X_train.columns, "Coefficient": ols.coef_})

#organization of top features affecting price
coef_df["Top Features"] = np.abs(coef_df["Coefficient"])
coef_df = coef_df.sort_values(by = "Top Features", ascending = False)
print("OLS MSE:", mse_ols)
print("RMSE:", rmse)
print("R^2:", r2)
print("Intercept:", ols.intercept_)
print("\n Top Features Affecting Housing Price:")
print(coef_df.head(5))

OLS MSE: 1771751116594.0352
RMSE: 1331071.4167895108
R^2: 0.6494754192267803
Intercept: -127711.16739244293

 Top Features Affecting Housing Price:
            Feature   Coefficient  Top Features
2         bathrooms  1.097117e+06  1.097117e+06
8   airconditioning  7.855506e+05  7.855506e+05
7   hotwaterheating  6.878813e+05  6.878813e+05
10         prefarea  6.299017e+05  6.299017e+05
3           stories  4.062232e+05  4.062232e+05
